In [ ]:
# Import the required packages
import os
import sys
import json
import time
from typing import Literal
from pydantic import BaseModel, Field
from openai import OpenAI

In [ ]:
# Load the results
with open("llm_results_qwen2.5_14B.json", "r", encoding="utf-8") as f:
    llm_results = json.load(f)

In [ ]:
# Load the EIOS data
with open("eios_historical_en.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [ ]:
# Delete additional keys
for result in llm_results.values():
    result.pop("pub_date", None)
    result.pop("url", None)
    result.pop("source_name", None)

In [ ]:
# Define Pydantic schema 
class CategoryEvaluation(BaseModel):
    status: Literal["Pass", "Fail"] = Field(
        description="'Pass' if the field follows core logic/grounding rules; otherwise 'Fail'."
    )
    comment: str = Field(
        description="Concise 1-sentence explanation citing exact text evidence."
    )


class JudgeEvaluation(BaseModel):
    label_and_outbreak_compliance: CategoryEvaluation = Field(
        description="Evaluation of primary label and outbreak_detected status."
    )
    entity_grounding_compliance: CategoryEvaluation = Field(
        description="Evaluation of country, ISO2, species, and event_date accuracy."
    )
    formatting_and_reasoning_compliance: CategoryEvaluation = Field(
        description="Evaluation of reasoning relevance and word counts."
    )
    total_rating: Literal[1, 2, 3] = Field(
        description="1 = Major classification failure; 2 = Correct label but minor entity/length issue; 3 = Flawless."
    )

In [ ]:
judge_prompt = """
You are an expert judge evaluating an epidemiological extraction system against the source text.

### Evaluation guidelines

#### 1. Label & outbreak rules
- label="epi_summary" & outbreak_detected=false: Used for macro surveillance, routine statistics, or multi-country tables.
- label="outbreak_alert" & outbreak_detected=true: Used for active, sudden, ongoing, or alarming local outbreaks.
- label="other" & outbreak_detected=false: Used for environmental/vector-only samples (e.g., positive mosquito traps) or general medical news.

#### 2. Entity grounding rules
- Countries & dates MUST be explicitly stated in the source text. No assumptions.
- ISO2 codes must accurately match mentioned countries.
- If label="epi_summary" and multiple countries are named, ALL mentioned countries must be extracted.
- If positive mosquito-only test: species_affected MUST be empty ([]).

#### 3. Formatting Rules
- Reasoning field must be relevant and concise (target <= 10 words).

### 4. Total rating rules
- 1 (Major failure): Incorrect 'label' or 'outbreak_detected' status, or major hallucinated facts.
- 2 (Minor breach): Core label and outbreak status are 100% correct, but missed an extracted country, wrong ISO2, or reasoning was too long.
- 3 (Flawless): Everything is 100% accurate, fully grounded, and strictly adheres to formatting.
"""

In [ ]:
llm_prompt = """

You are an epidemiologist classifying West Nile virus articles.

Return ONE JSON object.

Article:
{article_text}

---

SCOPE RULE (VERY IMPORTANT):
Only consider West Nile virus (WNV).

If the article is about ANY other disease → classify as "other".

---

CLASSIFICATION RULES (apply in order):

1) If there are NO human or animal infections mentioned
(only mosquito traps/pools/surveillance, no infected humans/animals, research on climate change etc):
→ label = "other"
→ outbreak_detected = false
→ species_affected = []
→ STOP

2) If the article is a surveillance report, statistics table, or seasonal summary:
→ label = "epi_summary"
→ outbreak_detected = false
*CRITERIA*: This includes reports capturing multi-national data, seasonal comparisons, or tracking cumulative statistics across multiple countries.
*EXAMPLE*: "WNV in Europe 2022: EU countries reported 292 cases across Italy (228), Greece (59), and Austria (2)." This is an epi_summary

3) If there are confirmed or suspected active, localized spikes, unexpected cases, or emergency notifications:
→ label = "outbreak_alert"
→ outbreak_detected = true
*CRITERIA*: Tone features real-time concern, unexpected increases, or immediate localized threats in a country/region.
*EXAMPLE*: "In recent weeks, increasingly alarming news has spread about the increase in cases of West Nile Disease in our country. The cases reported in Italy by the National Reference Center for WND, at the Zooprophylactic Institute, have risen to 230..." This must be an outbreak_alert with outbreak_detected = true.

1) If there are NO human or animal infections mentioned
(only mosquito traps/pools/surveillance, no infected humans/animals):
→ classify = "other"
→ outbreak_detected = false
→ species_affected = []
→ STOP

2) If the article is a surveillance report, statistics, or seasonal summary:
→ classify = "epi_summary"

3) If there are confirmed or suspected human/animal cases:
→ classify = "outbreak_alert"

---

CRITICAL RULES:
- outbreak_detected = true ONLY if the text explicitly mentions cases, infections or outbreak.
- If only research, modelling, risk, or discussion → outbreak_detected = false
- Mosquito-only positivity WITHOUT human/animal cases is ALWAYS "other" → outbreak_detected = false
- ONLY extract information explicitly stated in the text.
- Do NOT infer, assume, or generalize.
- Do NOT guess countries or species if not explicitly mentioned.
- If unsure, return empty list [] and false.
- If no specific event date is mentioned in the text, set "event_date" to null.
- Reason must be maximum 10 words, do not exceed 10 words under any circumstance

Return ONLY valid JSON.

JSON:"""

In [ ]:
# Initialize client
client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"]) 

In [ ]:
%%time
output_file = "gpt4o_evaluation_qwen2.5_14B.jsonl"

# Quickly scan what we have already saved on disk before starting
existing_ids = set()
if os.path.exists(output_file):
    with open(output_file, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                try:
                    record = json.loads(line)
                    if "article_id" in record:
                        existing_ids.add(record["article_id"])
                except json.JSONDecodeError:
                    continue

print(f"Loaded {len(existing_ids)} already processed records. Starting evaluation loop...")

# Main processing loop
for item in data:
    article_id = item["id"]

    # Check if we should skip this item
    if article_id in existing_ids:
        continue

    formatted_llm_prompt = llm_prompt.format(
        article_text=item["en_text"]
    )
    result = llm_results.get(article_id)

    user_dynamic_content = f"""### Input Data
[Target LLM prompt / constraints]:
{formatted_llm_prompt}

[Target LLM output]:
{result}"""

    try:
        #  API call
        chat_completion = client.beta.chat.completions.parse(
            model="gpt-4o-mini",
            messages=[
                {"role": "developer", "content": judge_prompt},
                {"role": "user", "content": user_dynamic_content}
            ],
            temperature=0.0,
            response_format=JudgeEvaluation
        )

        # Extract the generated JSON text
        raw_json_string = chat_completion.choices[0].message.content

        # Parse into dict and bind the key article identifier
        saved_record = json.loads(raw_json_string)
        saved_record["article_id"] = article_id

        # Append to the file 
        with open(output_file, "a", encoding="utf-8") as f:
            f.write(json.dumps(saved_record) + "\n")

        print(f"Saved article id {article_id} successfully. Total rating: {saved_record['total_rating']}")

        # Keep a steady internal pace
        time.sleep(0.2)

    except Exception as e:
        # Catch unexpected errors
        print(f"\nError for article id {article_id}: {e}")
        print("All data processed up to this point is safe.")
        break